In [1]:
# Install CacheFlow Library
!pip install ./cacheflow-0.1.0-py3-none-any.whl

Processing ./cacheflow-0.1.0-py3-none-any.whl


In [2]:
# Install dependencies
!pip install transformers accelerate safetensors rich pandas matplotlib seaborn

import torch
import time
import rich
from transformers import AutoModelForCausalLM, AutoTokenizer
from cacheflow import cacheflow_optimizer

# Set device to Colab's T4 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Environment Ready. Utilizing Device: {device.upper()}")

✅ Environment Ready. Utilizing Device: CUDA


In [3]:
model_path = "ibm-granite/granite-3.1-3b-a800m-instruct"
print(f"📥 Loading Baseline Model: {model_path}...")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)

if torch.cuda.is_available():
    torch.cuda.synchronize()
    baseline_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print("\n" + "="*50)
    print(f"🔴 BASELINE PEAK VRAM: {baseline_vram:.2f} MB")
    print(f"Status: Dangerously close to edge-device limits.")
    print("="*50)

📥 Loading Baseline Model: ibm-granite/granite-3.1-3b-a800m-instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


🔴 BASELINE PEAK VRAM: 6292.00 MB
Status: Dangerously close to edge-device limits.


In [4]:
print("🚀 Applying CacheFlow Software-Level LRU Scheduler...")

# We set capacity to 3 and enable the live terminal logging
model = cacheflow_optimizer(model, num_gpu_slots=3, presentation_mode=True)

if torch.cuda.is_available():
    torch.cuda.synchronize()
    optimized_vram = torch.cuda.memory_allocated() / (1024 ** 2)
    print("\n" + "="*50)
    print(f"🟢 CACHEFLOW ACTIVE VRAM: {optimized_vram:.2f} MB")
    print(f"Status: Highly Optimized. Ready for Edge Deployment.")
    print("="*50)

🚀 Applying CacheFlow Software-Level LRU Scheduler...

🚀 CacheFlow Optimizer Applied Successfully!
✅ Replaced 64 Standard MoE blocks with CacheFlow Dynamic Schedulers.
✅ Capacity limited to 3 Active Experts per layer.

🟢 CACHEFLOW ACTIVE VRAM: 6291.95 MB
Status: Highly Optimized. Ready for Edge Deployment.


In [5]:
input_text = "Briefly explain how Mixture-of-Experts (MoE) models work."
input_tokens = {k: v.to(device) for k, v in tokenizer(input_text, return_tensors="pt").items()}

print("🧠 Initiating Generation Sequence...\n")

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

# Watch the terminal for the live LRU Cache Hits/Misses!
output = model.generate(**input_tokens, max_new_tokens=60)

end_time = time.time()
speed = output.shape[1] / (end_time - start_time)

print("\n" + "="*50)
print(f"⚡ INFERENCE COMPLETE")
print(f"⚡ Speed: {speed:.2f} Tokens/Sec")
print("="*50 + "\n")

rich.print(tokenizer.batch_decode(output)[0])

🧠 Initiating Generation Sequence...


⚡ INFERENCE COMPLETE
⚡ Speed: 2.37 Tokens/Sec



Briefly explain how Mixture-of-Experts (MoE) models work.

Mixture-of-Experts (MoE) models are a type of deep learning architecture that combines the strengths of multiple 
expert models to improve overall performance. Here's a brief explanation of how they work:

1. **Expert Models**: MoE models consist of